In [ ]:
#Extract packages
import pandas as pd
import numpy as np

In [ ]:
#Extract dataset
df = pd.read_csv('https://raw.githubusercontent.com/HumayDS/A15---24-Reqemsal-data-analitika/refs/heads/main/user_activity_growth_2.csv')

In [ ]:
#date – tarix
#new_users – yeni istifadəçilər
#active_users – aktiv istifadəçilər
#sessions – sessiya sayı
#acquisition_channel - müştəri hansı kanaldan gəlib
#avg_session_duration – orta sessiya müddəti (dəqiqə)
df.head()

In [ ]:
#Kateqooriya analizi
df['acquisition_channel'].unique()

In [ ]:
#KAteqoriyalari eynilesdir
df['acquisition_channel'] = df['acquisition_channel'].replace({
    'socialmedia': 'social_media'
})

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df = df.fillna(df.median(numeric_only=True))

In [ ]:
df.isnull().sum()

In [ ]:
#Dublikat check
df.duplicated().sum()

In [ ]:
#Dublikat check
df.groupby(['date', 'acquisition_channel']).size().reset_index(name='count').query('count > 1')

In [ ]:
#Dublikatları sil
df = df.drop_duplicates()

In [ ]:
df.info()

In [ ]:
#Time formatina cevir
df['date'] = pd.to_datetime(df['date'])
df.info()

In [ ]:
df.describe().T

In [ ]:
#Baslangic ve son dovre baxaq
start_date = df['date'].min()
end_date = df['date'].max()

print(f'Start Date: {start_date}')
print(f'End Date: {end_date}')

In [ ]:
#AY ve il sutunu yarat
df['month'] =  df['date'].dt.month
df['year'] = df['date'].dt.year

In [ ]:
#Outlier check
num_cols = ['new_users', 'active_users', 'sessions', 'avg_session_duration']
Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1

outlier_mask = (df[num_cols] < (Q1 - 2 * IQR)) | (df[num_cols] > (Q3 + 2 * IQR))

In [ ]:
#Outlier check
outlier_mask.sum()

In [ ]:
df_outliers = df[outlier_mask.any(axis=1)]
df_outliers.head(20)

# Qayda
#Outlier 2 cür olur : Real event(silinməməlidir) , Data error (silinməlidir)
#Outlier silməzdən əvvəl səbəbini izah edə bilməlisən.
#Əgər izah edə bilmirsənsə:

##ya saxla
##ya ayrıca flag et

In [ ]:
df['is_outlier'] = outlier_mask.any(axis=1)

In [ ]:
df.describe().T

In [ ]:
#Outlieri sil
df_clean = df[~outlier_mask.any(axis=1)]

In [ ]:
df_clean.describe().T

In [ ]:
new_users_by_year_month = df.groupby(['year', 'month'])['new_users'].sum()
print(new_users_by_year_month)

In [ ]:
#Kanal uzre analitika
channel_users = df.groupby('acquisition_channel')['active_users'].sum().reset_index()
channel_users = channel_users.sort_values(by='active_users', ascending=False)
channel_users

In [ ]:
new_users_by_year_month = df.groupby(['year', 'month'])['new_users'].sum()
print(new_users_by_year_month)

In [ ]:
df.head()

In [ ]:
dau = df.groupby('date')['active_users'].sum().reset_index()
dau

In [ ]:
#1. DAU (Daily Active Users trend)
dau['growth_rate'] = dau['active_users'].pct_change()

In [ ]:
#hansı günlərdə sürətli artım / düşüş var
dau

In [ ]:
#Ən sürətli artım günlərini tapaq
dau.sort_values(by='growth_rate', ascending=False).head(5)

In [ ]:
#Araşdırılmalı günlər
# +20% və yuxarı artım
high_growth = dau[dau['growth_rate'] > 0.2]

# -20% və aşağı düşüş
high_drop = dau[dau['growth_rate'] < -0.2]

In [ ]:
high_growth

In [ ]:
#. Sessions per user --- 1 istifadəçi orta hesabla neçə sessiya edir
df['sessions_per_user'] = df['sessions'] / df['active_users']
df.head()

In [ ]:
channel_sessions_per_user = df.groupby('acquisition_channel')['sessions_per_user'].mean().reset_index()
print(channel_sessions_per_user)


In [ ]:
#Total engagement
df['total_time'] = df['sessions'] * df['avg_session_duration']

In [ ]:
df.head()

In [ ]:
#Kanal performansi
channel_perf = df.groupby('acquisition_channel').agg(
    total_time=('total_time', 'sum'),
    count=('date', 'count')
).reset_index()

channel_perf = channel_perf.sort_values(by='total_time', ascending=False)

channel_perf

In [ ]:
#Heftenin gunleri uzre ortalama session
df['weekday'] = pd.to_datetime(df['date']).dt.day_name()

df.groupby('weekday')['sessions'].mean()

In [ ]:
#Anomaly days
df[df['is_outlier'] == True]

# Ən çox istifadəçi gətirən kanal ən effektiv kanal deyil — effektivlik istifadəçinin platformada keçirdiyi vaxt və aktivliklə ölçülməlidir.